# Cad West, Wales — Terrain Processing Pipeline Demo

Demonstrates the full LiteAeroSim terrain ingestion pipeline for a 10 km × 10 km grid
centred on **Cad West, Cambrian Mountains, Wales (52.6983°N, 3.8691°W)**:

1. Generate (or download) a DEM and an imagery raster for the area
2. Triangulate to an LOD 0 TIN (≈ 10 m vertex spacing) — full resolution
3. Simplify to LOD 1 (≈ 30 m) and LOD 2 (≈ 100 m)
4. Colorize each LOD by sampling the imagery raster (`colorize.py`)
5. Export to `.las_terrain` binary format
6. Visualise all three LODs with PyVista — interactive, with daylight sun illumination

**Real data download is optional.** The notebook runs fully offline using a synthetic
heightfield and a synthetic Sentinel-2-style imagery raster at matching 10 m resolution.

## 1 — Imports and path setup

In [ ]:
import sys
import math
import time
import tempfile
from pathlib import Path

import numpy as np
from scipy.ndimage import gaussian_filter
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_bounds

# Notebook lives in python/; tools are in python/tools/ and python/tools/terrain/.
_HERE = Path(".").resolve()
for _p in [str(_HERE / "tools"), str(_HERE / "tools" / "terrain")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from las_terrain import TerrainTileData, read_las_terrain, write_las_terrain
from triangulate import triangulate
from simplify import simplify
from colorize import colorize
from verify import check, MeshQualityError
from export import export_las_terrain

print("Terrain tools loaded.")

## 3 — Synthetic DEM and imagery generation

Two rasters are generated and saved as GeoTIFFs so the standard pipeline tools can read them:

- **DEM** — float32, single band, EPSG:4326, elevations in metres (fBm octave noise);
  pixel spacing ≈ 10 m to match LOD 0 grid spacing
- **Imagery** — int16, 4 bands, EPSG:4326, Sentinel-2 surface reflectance × 10 000
  at matching 10 m resolution (bands: 1 = unused, 2 = Blue, 3 = Green, 4 = Red —
  matching `colorize.py` `source='sentinel2'`)

The imagery is intentionally decorrelated from the DEM (different noise seed, added spatial
texture) to simulate the independence of real elevation and optical data sources.

**Skip this cell and run §4 instead** if you have real Copernicus/Sentinel-2 tiles.

In [ ]:
# Cad West, Cambrian Mountains, Wales
CAD_WEST_LAT_DEG = 52.698332   # degrees N
CAD_WEST_LON_DEG = -3.869144   # degrees E (negative = West)

# 10 km x 10 km grid: +/-5 km from centroid
HALF_EXTENT_KM = 5.0
lat_half_deg = HALF_EXTENT_KM / 111.1
lon_half_deg = HALF_EXTENT_KM / (111.1 * math.cos(math.radians(CAD_WEST_LAT_DEG)))

# (lon_min, lat_min, lon_max, lat_max) — bbox_deg convention used by triangulate.py
BBOX_DEG = (
    CAD_WEST_LON_DEG - lon_half_deg,
    CAD_WEST_LAT_DEG - lat_half_deg,
    CAD_WEST_LON_DEG + lon_half_deg,
    CAD_WEST_LAT_DEG + lat_half_deg,
)
lon_min, lat_min, lon_max, lat_max = BBOX_DEG

print(f"Centroid : {CAD_WEST_LAT_DEG:.6f} N, {abs(CAD_WEST_LON_DEG):.6f} W")
print(f"Bbox     : [{lon_min:.5f}, {lat_min:.5f}] -> [{lon_max:.5f}, {lat_max:.5f}] deg")
print(f"Grid     : {2*HALF_EXTENT_KM:.0f} km x {2*HALF_EXTENT_KM:.0f} km")

## 3 — Synthetic DEM and imagery generation

Two rasters are generated and saved as GeoTIFFs so the standard pipeline tools can read them:

- **DEM** — float32, single band, EPSG:4326, elevations in metres (fBm octave noise)
- **Imagery** — int16, 4 bands, EPSG:4326, Sentinel-2 surface reflectance × 10 000
  (bands: 1 = unused, 2 = Blue, 3 = Green, 4 = Red — matching `colorize.py` `source='sentinel2'`)

The imagery is intentionally decorrelated from the DEM (different noise seed, added spatial
texture) to simulate the independence of real elevation and optical data sources.

**Skip this cell and run §4 instead** if you have real Copernicus/Sentinel-2 tiles.

In [ ]:
# ---- DEM -----------------------------------------------------------------------

def generate_welsh_terrain(ny: int, nx: int, seed: int = 42) -> np.ndarray:
    """Synthetic Welsh upland terrain via octave-summed Gaussian noise (fBm approximation).

    Sigma values are tuned for a 10 m/pixel raster over a 10 km domain:
      - Large octaves (sigma ~400-150 px): plateaux and valley systems (~4-1.5 km scale)
      - Mid octaves (sigma ~60-25 px): ridgelines and cwms (~600-250 m scale)
      - Fine octaves (sigma ~10-2 px): surface roughness (~100-20 m scale)

    Returns a (ny, nx) float32 array with elevation in metres.
    Typical Cad West area: 280–760 m ASL, blanket bog and moorland.
    """
    rng = np.random.default_rng(seed)
    h = np.zeros((ny, nx), dtype=np.float64)
    for sigma, amp in [(400,1.00),(150,0.60),(60,0.35),(25,0.20),(10,0.12),(4,0.07),(2,0.04)]:
        raw = rng.standard_normal((ny, nx))
        smoothed = gaussian_filter(raw, sigma=sigma)
        s = smoothed.std()
        if s > 1e-10:
            smoothed /= s
        h += amp * smoothed
    y_n = np.linspace(-1.0, 1.0, ny)
    x_n = np.linspace(-1.0, 1.0, nx)
    xx, yy = np.meshgrid(x_n, y_n)
    h += (
        0.65 * np.exp(-((xx - 0.10) ** 2 / 0.20  + (yy + 0.05) ** 2 / 0.15))
        + 0.30 * np.exp(-((xx + 0.40) ** 2 / 0.12 + (yy - 0.35) ** 2 / 0.10))
    )
    h -= h.min()
    h = h / h.max() * 480.0 + 280.0  # 280–760 m ASL (Cambrian Mountains range)
    return h.astype(np.float32)


# ---- Imagery -------------------------------------------------------------------

# Photometric encoding parameters — must match colorize.py DISPLAY_GAIN/DISPLAY_GAMMA.
# colorize.py applies: display_uint8 = (DN/10000 * gain)^(1/gamma) * 255
# Inverse (target uint8 T → DN): DN = (T/255)^gamma * 10000 / gain
_S2_GAIN  = 3.5   # matches colorize.DISPLAY_GAIN["sentinel2"]
_S2_GAMMA = 2.2   # matches colorize.DISPLAY_GAMMA["sentinel2"]

def _encode_s2(target_u8: np.ndarray) -> np.ndarray:
    """Convert target display uint8 values to Sentinel-2 DN (reflectance × 10000)."""
    return (target_u8 / 255.0) ** _S2_GAMMA * 10000.0 / _S2_GAIN

# Land-cover palette: target display uint8 values at each elevation stop.
# These represent the desired on-screen appearance after gain + gamma correction.
_IMG_STOPS = np.array([0.00, 0.15, 0.38, 0.60, 0.82, 1.00])
_IMG_PALETTE_U8 = np.array([
    [ 74, 155,  55],  # valley floor  — improved pasture / woodland
    [110, 160,  65],  # lower slope   — rough grazing
    [165, 148,  85],  # upland        — moorland / bracken
    [148, 118,  75],  # high upland   — blanket peat
    [180, 155, 110],  # near-summit   — bare peat and rock
    [210, 205, 195],  # summit        — exposed rock and scree
], dtype=np.float64)
_IMG_PALETTE_S2 = _encode_s2(_IMG_PALETTE_U8)  # palette in Sentinel-2 DN units


def generate_synthetic_imagery(dem_arr: np.ndarray, ny: int, nx: int,
                                seed: int = 99) -> np.ndarray:
    """Generate a synthetic 4-band Sentinel-2-style surface reflectance raster.

    Band layout (1-indexed, as used by rasterio):
      Band 1 — unused placeholder
      Band 2 — Blue  reflectance × 10000
      Band 3 — Green reflectance × 10000
      Band 4 — Red   reflectance × 10000

    DN values are encoded so that applying colorize.py's gain+gamma pipeline
    reproduces the _IMG_PALETTE_U8 target display colors.

    Returns a (4, ny, nx) int16 array.
    """
    rng = np.random.default_rng(seed)

    # Normalise DEM to [0, 1], then decorrelate with independent spatial texture.
    e = (dem_arr - dem_arr.min()) / (dem_arr.max() - dem_arr.min())
    texture = gaussian_filter(rng.standard_normal((ny, nx)), sigma=30)
    texture /= texture.std() + 1e-10
    texture *= 0.09  # ±9 % perturbation — introduces land-cover variation
    e_tex = np.clip(e + texture, 0.0, 1.0)

    R = np.zeros((ny, nx), dtype=np.float64)
    G = np.zeros((ny, nx), dtype=np.float64)
    B = np.zeros((ny, nx), dtype=np.float64)

    for seg in range(len(_IMG_STOPS) - 1):
        lo, hi = _IMG_STOPS[seg], _IMG_STOPS[seg + 1]
        mask = (e_tex >= lo) & (e_tex <= hi)
        if not mask.any():
            continue
        frac = (e_tex[mask] - lo) / (hi - lo)
        for arr, ch in [(R, 0), (G, 1), (B, 2)]:
            arr[mask] = (
                _IMG_PALETTE_S2[seg, ch] * (1.0 - frac)
                + _IMG_PALETTE_S2[seg + 1, ch] * frac
            )

    # Stack into (4, ny, nx): band1=placeholder, band2=B, band3=G, band4=R
    return np.stack([
        B.astype(np.int16),  # band 1 — placeholder (unused by sentinel2 config)
        B.astype(np.int16),  # band 2 — Blue
        G.astype(np.int16),  # band 3 — Green
        R.astype(np.int16),  # band 4 — Red
    ], axis=0)


# ---- Generate and save ---------------------------------------------------------

# 10 m/pixel raster — matches LOD 0 grid spacing (0.000090 deg ≈ 10 m).
DEM_NY = round((lat_max - lat_min) / 0.000090)
DEM_NX = round((lon_max - lon_min) / 0.000090)
_TMP_DIR = Path(tempfile.mkdtemp(prefix="cad_west_"))
DEM_PATH = _TMP_DIR / "dem_synthetic.tif"
IMAGERY_PATH = _TMP_DIR / "imagery_synthetic_s2.tif"

tfm = from_bounds(lon_min, lat_min, lon_max, lat_max, DEM_NX, DEM_NY)

print(f"Generating {DEM_NX}x{DEM_NY} DEM (~10 m/px) ...", end=" ", flush=True)
t0 = time.time()
dem_array = generate_welsh_terrain(DEM_NY, DEM_NX)
with rasterio.open(DEM_PATH, "w", driver="GTiff",
                   height=DEM_NY, width=DEM_NX, count=1,
                   dtype="float32", crs=CRS.from_epsg(4326), transform=tfm) as dst:
    dst.write(dem_array, 1)
print(f"done in {time.time()-t0:.1f}s  (elev {dem_array.min():.0f}-{dem_array.max():.0f} m)")

print(f"Generating {DEM_NX}x{DEM_NY} Sentinel-2-style imagery (~10 m/px) ...", end=" ", flush=True)
t0 = time.time()
img_array = generate_synthetic_imagery(dem_array, DEM_NY, DEM_NX)
with rasterio.open(IMAGERY_PATH, "w", driver="GTiff",
                   height=DEM_NY, width=DEM_NX, count=4,
                   dtype="int16", crs=CRS.from_epsg(4326), transform=tfm) as dst:
    dst.write(img_array)
print(f"done in {time.time()-t0:.1f}s")

print(f"  DEM:     {DEM_PATH}")
print(f"  Imagery: {IMAGERY_PATH}")

## 4 — (Optional) Real data download

Set `USE_REAL_DATA = True` and configure credentials to download authentic terrain data.

**Copernicus DEM GLO-30** (30 m, already ellipsoidal — no geoid correction needed):
- Register at <https://dataspace.copernicus.eu/> and create an OAuth2 client
- Export: `COPERNICUS_CLIENT_ID` and `COPERNICUS_CLIENT_SECRET`

**Sentinel-2 L2A imagery** (10 m true colour, surface reflectance × 10 000):
- Credentials as above (same Copernicus account)
- Set `IMAGERY_SOURCE = "sentinel2"` below

If credentials are absent the notebook continues with the synthetic rasters from §3.

In [ ]:
USE_REAL_DATA = True  # <- set True if credentials are configured
IMAGERY_SOURCE = "sentinel2"  # sentinel2 | landsat9 | modis

if USE_REAL_DATA:
    from download import download_dem, download_imagery, default_cache_dir, DownloadError
    from mosaic import mosaic_dem, mosaic_imagery

    print(f"Tile cache: {default_cache_dir()}")

    try:
        # output_dir omitted — tiles are cached to default_cache_dir()/dem/<source>/
        # and default_cache_dir()/imagery/<source>/ automatically.
        # Re-running over the same area skips the network entirely.
        print("Downloading Copernicus DEM tiles (cached per tile) ...")
        dem_tile_paths = download_dem(BBOX_DEG, source="copernicus_dem_glo30")
        mosaic_dem(dem_tile_paths, _TMP_DIR / "dem_real.tif")
        DEM_PATH = _TMP_DIR / "dem_real.tif"
        print(f"  DEM: {DEM_PATH}")

        print(f"Downloading {IMAGERY_SOURCE} imagery tiles (cached per tile) ...")
        img_tile_paths = download_imagery(BBOX_DEG, source=IMAGERY_SOURCE)
        mosaic_imagery(img_tile_paths, _TMP_DIR / "imagery_real.tif")
        IMAGERY_PATH = _TMP_DIR / "imagery_real.tif"
        print(f"  Imagery: {IMAGERY_PATH}")

    except Exception as exc:
        print(f"Download failed ({exc}); continuing with synthetic rasters.")
else:
    print("USE_REAL_DATA = False — using synthetic rasters from §3.")

print(f"Active DEM:     {DEM_PATH}")
print(f"Active imagery: {IMAGERY_PATH}")

## 5 — Triangulate at LOD 0 (≈ 10 m vertex spacing)

`triangulate.py` samples the DEM on a regular lon/lat grid at the LOD 0 spacing
(0.000090 ° ≈ 10 m) and runs `scipy.spatial.Delaunay` to build the base TIN.
Over the 10 km × 10 km bbox this produces roughly 1 000 × 1 660 = **1.66 M vertices**
and **3.3 M triangles** — the full-resolution mesh.

In [ ]:
print("Triangulating LOD 0 (0.000090 deg spacing, ~10 m) ...", end=" ", flush=True)
t0 = time.time()
tile_l0 = triangulate(DEM_PATH, BBOX_DEG, lod=0)
dt = time.time() - t0
nv0 = len(tile_l0.vertices)
nf0 = len(tile_l0.indices)
print(f"done in {dt:.1f}s")
print(f"  LOD 0: {nv0:,} vertices, {nf0:,} triangles")

## 6 — LOD simplification: LOD 0 → 1 → 2

`simplify.py` uses **pyfqmr QEM decimation** with `preserve_border=True` so that adjacent
tiles share crack-free edges. The target face count is scaled by the square of the spacing
ratio so that surface density is approximately consistent across LODs.

| LOD | Spacing | Target triangles |
|-----|---------|------------------|
| 0   | ≈  10 m | ~3.3 M           |
| 1   | ≈  30 m | ~370 k           |
| 2   | ≈ 100 m | ~41 k            |

In [ ]:
print("Simplifying LOD 0 -> LOD 1 ...", end=" ", flush=True)
t0 = time.time()
tile_l1 = simplify(tile_l0, target_lod=1)
nf1 = len(tile_l1.indices)
print(f"done in {time.time()-t0:.1f}s  ({nf1:,} triangles)")

print("Simplifying LOD 1 -> LOD 2 ...", end=" ", flush=True)
t0 = time.time()
tile_l2 = simplify(tile_l1, target_lod=2)
nf2 = len(tile_l2.indices)
print(f"done in {time.time()-t0:.1f}s  ({nf2:,} triangles)")

print()
print(f"Reduction ratios:  L0->L1 {nf0/nf1:.0f}x,  L1->L2 {nf1/nf2:.0f}x")

## 7 — Colorization from imagery texture map

`colorize.py` assigns per-facet RGB to each tile by sampling the imagery raster at each
facet's centroid geodetic position:

1. Compute facet centroid ENU offset → geodetic lon/lat (first-order approximation)
2. Sample the imagery raster with `rasterio.DatasetReader.sample()`
3. Extract Red (band 4), Green (band 3), Blue (band 2) per `BAND_ORDER["sentinel2"]`
4. Scale: `uint8 = clip(raw × (1/10000) × 255, 0, 255)`
5. Nodata pixels fall back to grey `{128, 128, 128}`

The same imagery path is used for all three LODs — colorize.py handles the
centroid-level sampling independently of mesh resolution.

In [ ]:
print("Colorizing LOD 0 from imagery ...", end=" ", flush=True)
t0 = time.time()
tile_l0_c = colorize(tile_l0, IMAGERY_PATH, source=IMAGERY_SOURCE)
print(f"done in {time.time()-t0:.2f}s")

print("Colorizing LOD 1 from imagery ...", end=" ", flush=True)
t0 = time.time()
tile_l1_c = colorize(tile_l1, IMAGERY_PATH, source=IMAGERY_SOURCE)
print(f"done in {time.time()-t0:.2f}s")

print("Colorizing LOD 2 from imagery ...", end=" ", flush=True)
t0 = time.time()
tile_l2_c = colorize(tile_l2, IMAGERY_PATH, source=IMAGERY_SOURCE)
print(f"done in {time.time()-t0:.2f}s")

## 7b — 2D texture map preview

Full-resolution color map read directly from the imagery raster.  This shows the
ground-truth colors that will be sampled by `colorize.py` — use it as a reference
to verify the 3D render is correct.

In [ ]:
import matplotlib.pyplot as plt
from colorize import DISPLAY_GAIN, DISPLAY_GAMMA, SCALE_FACTORS

# Apply the same gain + gamma pipeline used by colorize.py for display.
_s2_scale     = SCALE_FACTORS["sentinel2"]
_s2_gain      = DISPLAY_GAIN["sentinel2"]
_s2_inv_gamma = 1.0 / DISPLAY_GAMMA["sentinel2"]

with rasterio.open(IMAGERY_PATH) as src:
    r_raw = src.read(4).astype(np.float32)  # band 4 = Red
    g_raw = src.read(3).astype(np.float32)  # band 3 = Green
    b_raw = src.read(2).astype(np.float32)  # band 2 = Blue
    bounds = src.bounds

def _s2_to_u8(raw: np.ndarray) -> np.ndarray:
    lin = np.clip(raw * _s2_scale * _s2_gain, 0.0, 1.0)
    return (np.power(lin, _s2_inv_gamma) * 255.0).astype(np.uint8)

r8, g8, b8 = _s2_to_u8(r_raw), _s2_to_u8(g_raw), _s2_to_u8(b_raw)
rgb_img = np.stack([r8, g8, b8], axis=2)

fig, ax = plt.subplots(figsize=(13, 8))
ax.imshow(
    rgb_img,
    extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    origin="upper",
    aspect="equal",
)
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")
ax.set_title(
    f"Imagery texture map — Cad West, Wales\n"
    f"(Sentinel-2 true colour, gain={_s2_gain}, γ={DISPLAY_GAMMA['sentinel2']}, "
    f"{rgb_img.shape[1]}×{rgb_img.shape[0]} px)"
)
ax.grid(True, alpha=0.3, linewidth=0.5)
plt.tight_layout()
plt.show()

print(f"Color range — R: [{r8.min()}, {r8.max()}]  "
      f"G: [{g8.min()}, {g8.max()}]  B: [{b8.min()}, {b8.max()}]")

## 8 — Export to `.las_terrain`

All three LOD tiles are packed into a single `.las_terrain` binary file.  A round-trip
read is performed to confirm the archive is valid.

In [ ]:
LAS_PATH = _TMP_DIR / "cad_west.las_terrain"
export_las_terrain([tile_l0_c, tile_l1_c, tile_l2_c], LAS_PATH)
size_mb = LAS_PATH.stat().st_size / 1e6
print(f"Exported {LAS_PATH.name}  ({size_mb:.1f} MB)")

# Round-trip verify
rt_tiles = read_las_terrain(LAS_PATH)
print(f"Read back {len(rt_tiles)} tiles:")
for t in rt_tiles:
    print(f"  LOD {t.lod}: {len(t.vertices):,} vertices, {len(t.indices):,} triangles")

## 9 — PyVista 3D visualisation

Three LODs displayed side-by-side with linked cameras.  Illumination uses two lights:

- **Sun** — warm white, 35 ° elevation, 220 ° geographic azimuth (SSW afternoon sun, typical UK)
- **Sky fill** — cool blue, soft ambient from directly above (open-sky diffuse)

**Vertical exaggeration = 5×** makes the modest Welsh relief clearly visible at this scale.

Rotate with left-click drag, zoom with scroll, pan with right-click drag.

> **Note:** `trame` backend provides server-side VTK rendering, which is required for
> `pl.enable_shadows()`. The `html`/vtk.js backend does not implement shadow maps.

In [ ]:
import pyvista as pv

# trame provides server-side VTK rendering, which is required for shadow maps.
# The ipykernel asyncio compatibility issue that forced us to use 'html' is resolved
# by ipykernel >= 6.29.5 (7.2.0 is now installed).
pv.set_jupyter_backend("trame")

VERT_SCALE = 1.0  # vertical exaggeration

# --- Daylight sun: 35 deg elevation, 220 deg geographic azimuth (SSW afternoon, UK) ---
# ENU: East = +x, North = +y, Up = +z
sun_el = math.radians(35)
sun_az = math.radians(220)
_sun_e = math.sin(sun_az) * math.cos(sun_el)
_sun_n = math.cos(sun_az) * math.cos(sun_el)
_sun_u = math.sin(sun_el)
SUN_DIST = 5.0e6  # 5000 km — effectively directional
SUN_POS = (_sun_e * SUN_DIST, _sun_n * SUN_DIST, _sun_u * SUN_DIST * VERT_SCALE)

print(f"Sun position (ENU scaled): ({_sun_e:.3f}, {_sun_n:.3f}, {_sun_u:.3f}) × {SUN_DIST:.0e} m")
print(f"Vertical exaggeration: {VERT_SCALE}x")


def tile_to_pv_mesh(tile: TerrainTileData, vert_scale: float = 1.0) -> pv.PolyData:
    """Convert TerrainTileData to pv.PolyData with per-cell RGB colors.

    ENU axes: East = +X, North = +Y, Up = +Z (scaled by vert_scale).
    Colors are normalised to float32 [0, 1] — required for correct rendering in both
    trame and html pyvista backends.
    """
    verts = tile.vertices.astype(np.float64).copy()
    verts[:, 2] *= vert_scale

    faces_in = tile.indices.astype(np.int64)
    n_f = len(faces_in)
    pv_faces = np.empty(n_f * 4, dtype=np.int64)
    pv_faces[0::4] = 3
    pv_faces[1::4] = faces_in[:, 0]
    pv_faces[2::4] = faces_in[:, 1]
    pv_faces[3::4] = faces_in[:, 2]

    mesh = pv.PolyData(verts, pv_faces)
    # Normalise uint8 [0, 255] → float32 [0, 1].  Both trame and html backends
    # expect float colours when rgb=True; uint8 is silently normalised differently
    # across VTK versions, which caused the white/yellow wash-out.
    mesh.cell_data["colors"] = tile.colors.astype(np.float32) / 255.0
    return mesh


lod_tiles = [tile_l0_c, tile_l1_c, tile_l2_c]
lod_labels = [
    f"LOD 0  |  10 m spacing   |  {nf0:,} faces",
    f"LOD 1  |  30 m spacing   |  {nf1:,} faces",
    f"LOD 2  |  100 m spacing  |  {nf2:,} faces",
]
lod_meshes = [tile_to_pv_mesh(t, VERT_SCALE) for t in lod_tiles]

In [ ]:
# --- 1x3 interactive plotter ---
pl = pv.Plotter(
    shape=(1, 3),
    window_size=(1500, 560),
    notebook=True,
    border=True,
    border_color="grey",
)
pl.set_background("#d8e8f0")

for col, (mesh, label) in enumerate(zip(lod_meshes, lod_labels)):
    pl.subplot(0, col)

    pl.add_mesh(
        mesh,
        scalars="colors",
        rgb=True,
        lighting=True,
        smooth_shading=False,
        show_scalar_bar=False,
    )

    # Remove pyvista's default headlight — it washes out the colors when combined
    # with the custom sun light.  We replace it with two explicit lights:
    #   1. Directional sun light  — primary illumination and shadow caster
    #   2. Ambient sky fill       — low-intensity cool-blue fill for shadowed faces
    pl.remove_all_lights()

    # Directional sun light (SSW afternoon, UK)
    pl.add_light(pv.Light(
        position=SUN_POS,
        focal_point=(0.0, 0.0, 0.0),
        light_type="scene light",
        intensity=0.85,
        color=[1.00, 0.97, 0.88],  # warm daylight
    ))

    # Ambient sky fill — keeps shadowed areas from going pure black
    pl.add_light(pv.Light(
        position=(0.0, 0.0, 5.0e6 * VERT_SCALE),
        focal_point=(0.0, 0.0, 0.0),
        light_type="scene light",
        intensity=0.25,
        color=[0.68, 0.80, 1.00],  # cool sky blue
    ))

    # Shadow mapping — terrain self-shadows valleys and north-facing slopes
    pl.enable_shadows()

    pl.add_text(label, position="upper_left", font_size=9, color="black")

    half = 5_000.0
    pl.camera.position = (-half * 0.5, -half * 1.3, half * 0.7 * VERT_SCALE)
    pl.camera.focal_point = (0.0, 0.0, dem_array.mean() * 0.5 * VERT_SCALE)
    pl.camera.up = (0.0, 0.0, 1.0)

pl.link_views()
pl.show()

## 10 — Individual LOD deep-dive

Renders a single LOD in a larger window for detailed inspection.
Change `LOD_INDEX` to 0 (L3), 1 (L4), or 2 (L5).

In [ ]:
LOD_INDEX = 1  # 0=L0, 1=L1, 2=L2

tile_sel = lod_tiles[LOD_INDEX]
mesh_sel = lod_meshes[LOD_INDEX]
label_sel = lod_labels[LOD_INDEX]

pl2 = pv.Plotter(window_size=(900, 600), notebook=True)
pl2.set_background("#d8e8f0")

pl2.add_mesh(
    mesh_sel,
    scalars="colors",
    rgb=True,
    lighting=True,
    smooth_shading=False,
    show_scalar_bar=False,
)

pl2.remove_all_lights()
pl2.add_light(pv.Light(
    position=SUN_POS,
    focal_point=(0.0, 0.0, 0.0),
    light_type="scene light",
    intensity=0.85,
    color=[1.00, 0.97, 0.88],
))
pl2.add_light(pv.Light(
    position=(0.0, 0.0, 5.0e6 * VERT_SCALE),
    focal_point=(0.0, 0.0, 0.0),
    light_type="scene light",
    intensity=0.25,
    color=[0.68, 0.80, 1.00],
))
pl2.enable_shadows()

pl2.add_text(label_sel, position="upper_left", font_size=10, color="black")
pl2.add_text(
    f"Cad West, Wales  |  10 km grid  |  {VERT_SCALE:.0f}x vert. exag.",
    position="lower_right",
    font_size=8,
    color="black",
)

half = 5_000.0
pl2.camera.position = (-half * 0.5, -half * 1.3, half * 0.7 * VERT_SCALE)
pl2.camera.focal_point = (0.0, 0.0, dem_array.mean() * 0.5 * VERT_SCALE)
pl2.camera.up = (0.0, 0.0, 1.0)

pl2.show()